# Kết quả thu thập và chất lượng dữ liệu

## Bao phủ 25.000 ID đầu vào

Các audit toàn dải xác nhận mỗi thành viên có đủ 5.000 record, không còn ID chưa ghi nhận và lookup chưa giải quyết. Bao phủ Discovery được đọc riêng với mức hoàn thiện source và kiểm định.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
sys.path.insert(0, str(ROOT / 'utils'))
from preliminary_results import (load_members, progress_rows, assignment_outcomes,
    quality_rows, error_groups, performance_rows, run_range, session_members, coverage_rows, corpus_metrics, snapshot_rows, percentage, paper_timings)
members = load_members(ROOT)
sessions = session_members(ROOT, members)
def table(rows):
    frame = pd.DataFrame(rows, dtype=object)
    frame = frame.where(frame.notna(), 'Chưa đủ bằng chứng')
    def number(value):
        if isinstance(value, float):
            return f'{value:,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
        if isinstance(value, int) and not isinstance(value, bool):
            return f'{value:,}'.replace(',', '.')
        return str(value)
    display(HTML(frame.map(number).to_html(index=False, escape=True, border=0, classes='dataframe')))


In [ ]:
table([{k:v for k,v in row.items() if k!='Đã chạy theo báo cáo nhóm'} for row in progress_rows(members)])
audits=[m['audit'] for m in members if m['verified']]
assigned=sum(a['assigned_input_count'] for a in audits)
discovered=sum(a['discovery_statuses'].get('success',0) for a in audits)
passed=sum(a['validation_pass_papers'] for a in audits)
failed=sum(a['validation_fail_papers'] for a in audits)
table([{'Chỉ số theo audit toàn nhóm':'Discovery thành công','Số paper':discovered,
        'Mẫu số':assigned,'Tỷ lệ (%)':percentage(discovered,assigned)},
       {'Chỉ số theo audit toàn nhóm':'Paper đạt validator','Số paper':passed,
        'Mẫu số':passed+failed,'Tỷ lệ (%)':percentage(passed,passed+failed)},
       {'Chỉ số theo audit toàn nhóm':'Paper chưa đạt validator','Số paper':failed,
        'Mẫu số':passed+failed,'Tỷ lệ (%)':percentage(failed,passed+failed)}])

## Source và corpus tích lũy

Số liệu sau lấy từ cumulative của cả năm thành viên, trên năm dải 5.000 ID. TV1 mới ghi đủ 5.000 Discovery thành công và 7.852 version; bản TV5 sau sửa lookup ghi 8.348 version. Dòng gộp cộng các snapshot đủ coverage, không cộng các lượt resume chồng lặp. Các snapshot khác thời điểm, không phải một lần chụp đồng thời toàn corpus.

Download success là trạng thái checkpoint. Source có tệp cần đúng đường dẫn và tệp khác rỗng; corpus có TeX/Bib cần extract thành công và có tệp phù hợp. Paper đủ source/corpus yêu cầu mọi version đã biết đáp ứng cả hai điều kiện.

In [ ]:
metrics=corpus_metrics(members)
table([{'Phạm vi':r['member'],'Paper Discovery':r['discovered'],'Version đã biết':r['versions'],
        'Download success (checkpoint)':r['download_success'],'Source xác minh có tệp':r['source_files'],
        'Version có corpus TeX/Bib':r['corpus_versions'],'Paper đủ mọi version':r['complete_papers']} for r in metrics])
table([{'Phạm vi':r['member'],'Discovery / ID được giao (%)':percentage(r['discovered'],r['assigned']),
        'Download success / version (%)':percentage(r['download_success'],r['versions']),
        'Source có tệp / version (%)':percentage(r['source_files'],r['versions']),
        'Paper đủ source-corpus / ID được giao (%)':percentage(r['complete_papers'],r['assigned'])} for r in metrics])
table([{'Thành viên':m['member'],'Version no_source':m['cumulative']['download_statuses'].get('no_source',0),
        'Version download api_error':m['cumulative']['download_statuses'].get('api_error',0)}
       for m in members if m['cumulative']])

Số paper thu thập thành công phụ thuộc tiêu chí: tìm được metadata, đủ source/corpus mọi version, hay đạt toàn bộ validator. Báo cáo công bố cả ba mức; Discovery 100% không có nghĩa corpus hoàn thiện 100%. No_source có thể là đầu ra hợp lệ cho PDF-only và được giữ riêng.

## Kích thước trước và sau lọc hình

Đơn vị là bytes/paper, cộng các version trong mỗi paper. Trước lọc dùng kích thước giải nén từ sidecar, sau lọc dùng tổng TeX/BibTeX được giữ; lệnh hình trong LaTeX không bị sửa. Chỉ paper có đủ cặp đo mới vào mẫu số. Trung bình gộp dùng trọng số số paper có cặp đo.

In [ ]:
table([{'Phạm vi':r['member'],'Paper có cặp đo':r['paired'],
        'Trước lọc (bytes/paper)':r['mean_before'],'Sau lọc (bytes/paper)':r['mean_after'],
        'Giảm dung lượng (%)':percentage(r['mean_before']-r['mean_after'],r['mean_before'])
            if r['mean_before'] is not None and r['mean_after'] is not None else None} for r in metrics])

## References và mức đầy đủ metadata

References đọc được gồm success hoặc no_data có JSON hợp lệ. Danh sách rỗng được tính vào mẫu số trung bình. Tỷ lệ đọc được lấy số paper Discovery thành công của cùng snapshot làm mẫu số, chưa gồm lookup lỗi.

Metadata đầy đủ yêu cầu cả năm trường ref_arxiv_id, ref_title, ref_authors, ref_submission_date và ref_s2_id có giá trị. Đây là độ đầy đủ theo quy tắc công cụ, không phải tỷ lệ API không lỗi hay độ chính xác ngữ nghĩa. References được đếm theo quan hệ paper–reference, loại trùng trong từng paper; không loại cùng tài liệu được nhiều paper trích dẫn.

In [ ]:
table([{'Phạm vi':r['member'],'References đã đọc':r['references'],'Paper có kết quả đọc được':r['retrieved'],
        'References/paper có kết quả':r['mean_references'],
        'Truy vấn có kết quả đọc được (%)':percentage(r['retrieved'],r['discovered']),
        'References đủ metadata':r['complete_references'],
        'Metadata đủ 5 trường (%)':percentage(r['complete_references'],r['reference_denominator'])} for r in metrics])
table([{'Phạm vi':r['member'],'Success':r['reference_statuses'].get('success',0),
        'No data':r['reference_statuses'].get('no_data',0),'Not indexed':r['reference_statuses'].get('not_indexed',0),
        'API/error khác':sum(v for k,v in r['reference_statuses'].items() if k not in {'success','no_data','not_indexed'})}
       for r in metrics])

TV2 có nhiều checkpoint references lỗi trong snapshot đã gửi; mức bao phủ thấp hơn các dải còn lại. Không suy số reference của paper lỗi bằng trung bình các paper đã truy vấn được.

## Chất lượng theo audit toàn dải

Tỷ lệ đạt dùng số paper đã kiểm định làm mẫu số. Audit và cumulative có thể khác ngày; validation cũ trong cumulative không thay audit mới.

In [ ]:
quality=quality_rows(members)
table(quality)
fig,ax=plt.subplots(figsize=(8,4))
ax.bar([r['Thành viên'] for r in quality],[r['Đạt'] for r in quality],label='Đạt kiểm định')
ax.bar([r['Thành viên'] for r in quality],[r['Còn lỗi'] for r in quality],
       bottom=[r['Đạt'] for r in quality],label='Chưa đạt')
ax.set_ylabel('Paper đã kiểm định');ax.legend();plt.tight_layout();plt.show()

Một paper chỉ đếm một lần trong từng nhóm lỗi; một paper có thể thuộc nhiều nhóm. Các nhóm không cộng thành tổng paper chưa đạt và chưa đủ để quy trách nhiệm cho một thành viên.

In [ ]:
table(error_groups(members))

## Thời điểm và phạm vi số liệu

Cumulative mới TV1 và audit đã nộp đều xác nhận đủ 5.000 Discovery thành công. TV5 cũng có cumulative sau sửa lookup. Validation vẫn lấy từ audit theo ngày kiểm tra, không coi các tệp cumulative là phép kiểm định mới. Các mốc UTC cho phép đối chiếu từng phép đọc.

In [ ]:
table(snapshot_rows(members))

### Đối chiếu trực tiếp dữ liệu TV1

Manifest trong [thư mục data TV1](https://drive.google.com/drive/folders/1uBRPdHxO2MTn2ahAkRtNSkxd4Rcu1DHv) được đọc ngày 10/11. Kiểm tra mẫu corpus 2508.17481/v1 thấy main.tex, references.bib và các tệp TeX khác. Đây là xác nhận Discovery và một mẫu corpus, chưa phải phép đo kích thước/references toàn TV1.

In [ ]:
import hashlib
folder=ROOT/'summaries/milestone1/drive-inspection-20261110/TV1'
receipt=json.loads((folder/'provenance.json').read_text(encoding='utf-8'))
for name,digest in receipt['files'].items():
    assert hashlib.sha256((folder/name).read_bytes()).hexdigest()==digest, 'TV1 Drive evidence changed'
inspection=json.loads((folder/'inspection.json').read_text(encoding='utf-8'))
table([{'Thành viên':'TV1','Trạng thái bao phủ':'Đã xác nhận trên Drive',
        'Discovery thành công':inspection['discovery_statuses']['success'],
        'Version trong manifest':inspection['versions'],
        'ID chưa ghi nhận':inspection['unseen_ids'],'ID ngoài dải':inspection['out_of_range']}])